In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

import numpy as np
import matplotlib

import matplotlib.pyplot as plt

import time
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
import umap
from sklearn.cluster import DBSCAN
from sklearn import metrics

from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as pl
import shap

import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="Corrs"

# Load and initialize

In [ ]:
import glob

In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/"

In [ ]:
FList=glob.glob(dir+"norm*")
FList.sort()
FList

In [ ]:
FList=['/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_11.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_13.parquet',
  '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_14.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_15.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_17.parquet',

 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_18.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_19.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_20.parquet',
  '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_4.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_5.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_7.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_8.parquet']

In [ ]:
Numbs=[11,13,14,15,17,18,19,20,4,5,7,8]

In [ ]:
len(Numbs)

In [ ]:
Numbs.sort()
Numbs

In [ ]:
DBs=[f"BCK{N}" for N in Numbs]
DBs

In [ ]:
for DB,N in zip(DBs,Numbs):
    print(DB)
    globals()[DB]=pd.read_parquet(f"/Users/ronguy/Dropbox/CyTOF_Breast/for_guy/normalized_not_scaled_{N}.parquet")

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
Rep

In [ ]:
for DB in DBs:
#    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)
    try:
        globals()[DB].drop(columns=['DNA1','DNA2','Event #'],inplace=True)
    except:
        pass

In [ ]:
N=list(globals()[DBs[0]].columns)
N.sort()
N.remove('N-cadherin')

In [ ]:
BCK8['N-cadherin']

In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)

In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
len(NamesAll)

In [ ]:
for DB in DBs:
    globals()[DB]=globals()[DB][(globals()[DB][NamesAll] >= 0).all(axis=1)]

In [ ]:
MRK_All=NamesAll.copy()
MRK_All.remove('H3')
MRK_All.remove('H3.3')
MRK_All.remove('H4')
#MRK_All.remove('H2A')

EPC=EpiCols.copy()
Core=['H3','H3.3','H4']#,'H2A']
for C in Core:
    EPC.remove(C)

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB]]).copy()


In [ ]:
NC=2000
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC,replace=False)]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
#    globals()[DB]=np.arcsinh(globals()[DB]/5)
    globals()[DB]['Line']=DB


params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")

In [ ]:
import distinctipy
DBCLR=dict(zip(DBs,distinctipy.get_colors(len(DBs))))

In [ ]:
N='H3K4me1'
ax = plt.subplot()
for DB in DBs:
    sns.histplot(data=globals()[DB],x=N,**hKWD,color=DBCLR[DB],bins=np.linspace(-20,10,200))

handles = [mpatches.Patch(color=DBCLR[db], label=db) for db in DBs]

ax.legend(
    handles=handles,
    title='Databases',
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    frameon=False          # drop the box if you don’t want one
);
plt.yscale('log')

In [ ]:
a=[]
fig, axs = plt.subplots(12, 3, figsize=(20, 20))
for i, row in enumerate(axs):
    for j, ax in enumerate(row):
        a.append(ax)
        
for i,N in enumerate(NamesAll):
    print(N)
    for DB in DBs:
        sns.histplot(data=globals()[DB],x=N,ax=a[i],**hKWD,color=DBCLR[DB])

#    a[i].set_yscale('log')
#    a[i].set_xscale('log')

#    a[i].set_title(N)

plt.subplots_adjust(wspace=0.5, hspace=1.9)
handles = [mpatches.Patch(color=DBCLR[db], label=db) for db in DBs]

# 2. Tell matplotlib to put the legend on the *right* side of the whole figure
#    - `bbox_to_anchor=(1.05, 0.5)` moves it a little outside the figure
#    - `loc='center left'` aligns the legend’s center‑left corner to that anchor
# 3. Use `fig.legend` so the legend is drawn **once** and not repeated on every subplot
fig.legend(
    handles=handles,
    title='Databases',
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    frameon=False          # drop the box if you don’t want one
)


In [ ]:
for DB in DBs:

    globals()[DB]['Line']=DB
    print(DB,globals()[DB].shape[0])

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB].sample(10000,replace=False)]).copy()


In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=None,verbose=True)

X_2d=UM.fit_transform(CAll[CellIden])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
CAll['x']=X_2d[:,0]
CAll['y']=X_2d[:,1]

In [ ]:
CAll.reset_index(drop=True,inplace=True)

In [ ]:
%matplotlib inline


In [ ]:
sns.scatterplot(data=CAll,x='x',y='y',hue='Line',s=1)
plt.show()

In [ ]:
%matplotlib widget

In [ ]:
gdf=ManualSelection(CAll)

In [ ]:
CAll=gdf.copy()

In [ ]:
CAll
%matplotlib inline

In [ ]:
for DB in DBs:
    M=CAll.Line==DB
    globals()[DB]=CAll[M].copy()
    globals()[DB]=globals()[DB][globals()[DB].region_id!=0]
    globals()[DB]=globals()[DB][NamesAll]
    

In [ ]:
for DB in DBs:

    globals()[DB]['Line']=DB
    print(DB,globals()[DB].shape[0])

In [ ]:
NC=2000
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC,replace=False)[NamesAll]]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
#    globals()[DB]=np.arcsinh(globals()[DB]/5)
    globals()[DB]['Line']=DB

In [ ]:
import distinctipy

In [ ]:
DBs

In [ ]:
NamesAll

In [ ]:
pKWD={'dpi':200,'bbox_inches':'tight'}

In [ ]:
%matplotlib inline

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB]]).copy()


In [ ]:
sns.histplot(data=CAll,x='H3K4me1',bins=np.linspace(-10,10,1000))
#plt.yscale('log')
plt.show()

In [ ]:
Mat=CAll.groupby('Line').mean()

In [ ]:
Mat=Mat.loc[DBs,:]

In [ ]:
Mat=Mat.astype(float)

In [ ]:
plt.figure(figsize=(15,10))
sns.heatmap(np.round(Mat[MRK_All].T,2),annot=True,cmap=plt.cm.seismic,center=0,yticklabels=True,xticklabels=True,)
plt.xticks(fontsize=12);
plt.yticks(fontsize=12);
#plt.savefig(f'Plots/{Run}_All.png',**pKWD)

In [ ]:
NC=1500
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB].sample(NC,replace=False)]).copy()
                  

In [ ]:
%matplotlib inline

In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=42,verbose=True)

X_2d=UM.fit_transform(CAll[MRK_All])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
plt.figure()
for i,L in enumerate(CAll.Line.unique()):
       
    M=CAll.Line==L
    
    plt.scatter(X_2d[M,0],X_2d[M,1],s=1,label=f"{L}",cmap=plt.cm.seismic,c=DBCLR[L])

    plt.legend(markerscale=10,fontsize=10)
plt.legend(bbox_to_anchor=(1,1),markerscale=10)
plt.show()

In [ ]:
import scanpy as sc
from sknetwork.clustering import Louvain, Leiden

In [ ]:
AN=sc.AnnData(CAll)
AN.obsm['X_umap']=X_2d


In [ ]:
vmn=[]
vmx=[]
for N in MRK_All:
    cc=CAll[N]
    v1,v2=cc.quantile(0.01),cc.quantile(0.99)
    vmn.append(v1)
    vmx.append(v2)

In [ ]:
sc.pl.umap(AN,color=MRK_All,cmap='seismic',vmin='p1',vmax='p99',show=False);


In [ ]:
EPIDat=pd.read_excel("EPINUC_Data/20250821_EPINUC_biomarkers_BCK_Cytof-samples_New.xlsx",index_col=0)

In [ ]:
EPIDat

In [ ]:
L=list(EPIDat.index)
L.sort()
L

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB]]).copy()


In [ ]:
Mat=CAll.groupby('Line').median(numeric_only=True)[MRK_All]

In [ ]:
Mat

In [ ]:
Dict1={
 'ER':'Count_ERObjects',
 'GATA3':'Count_GATA3Objects',
 'H3K27me3':'H3K27me3_Nuc',
 'H3K36me3':'H3K36me3_Nuc',
 'H3K4me1':'H3K4me1_Nuc',
 'H3K4me3':'H3K4me3_Nuc.WIS',
 'H3K9me2':'H3K9me2_Nuc',
 'H3K9ac':'H3K9ac_Nuc',
 'H3K9me3':'H3K9me3_Nuc',
 'H4K16ac':'H4K16ac_Nuc',
 'H4K20me3':'H4K20me3_Nuc',
 'KRT5':'Count_KRT5Objects',
}

In [ ]:
CyKeys=[f for f in Dict1.keys()]

In [ ]:
CombDB=pd.merge(EPIDat,Mat,left_index=True,right_index=True)

In [ ]:
CLM=list(CombDB.columns)
CLM=[s.replace("nuc", "Nuc") for s in CLM]
CombDB.columns=CLM

In [ ]:
# CombDB=CombDB[~(CombDB.index=='BCK11')]
# CombDB=(CombDB-CombDB.mean(axis=0))/CombDB.std(axis=0)

In [ ]:
#

In [ ]:
CombDB.to_csv("CombDB.csv",index=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def bland_altman(MRK, title=f"Bland–Altman Plot"):
    x = CombDB[Dict1[MRK]].values
    y = CombDB[MRK].values

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    mean = (x + y) / 2
    diff = x - y
    bias = np.mean(diff)
    sd = np.std(diff, ddof=1)
    loa_upper = bias + 1.96*sd
    loa_lower = bias - 1.96*sd

    plt.scatter(mean, diff, alpha=0.6)
    plt.axhline(bias, color='red', linestyle='--', label=f'Bias={bias:.2f}')
    plt.axhline(loa_upper, color='gray', linestyle='--', label=f'+1.96 SD={loa_upper:.2f}')
    plt.axhline(loa_lower, color='gray', linestyle='--', label=f'-1.96 SD={loa_lower:.2f}')
    plt.xlabel('Mean of methods')
    plt.ylabel('Difference (Method1 - Method2)')
    plt.title(title)
    plt.legend()
    plt.show()

    return {"bias": bias, "loa_lower": loa_lower, "loa_upper": loa_upper}

# Example:
# bland_altman(epinuc_vals, cytof_vals)
#bland_altman

In [ ]:
bland_altman('H3K4me3')

In [ ]:
CombDB["H3K4me3_Nuc.WIS"]

In [ ]:
from sklearn.linear_model import TheilSenRegressor
from tqdm import tqdm

In [ ]:
import pandas as pd
from scipy.stats import pearsonr,spearmanr

import numpy as np
from scipy.stats import theilslopes


import numpy as np
from sklearn.linear_model import HuberRegressor


import numpy as np
from sklearn.linear_model import RANSACRegressor, LinearRegression

def ransac_with_pvalue(x, y, n_permutations=1000, random_state=None, **ransac_kwargs):
    """
    RANSAC regression with permutation-based p-value for slope != 0.

    Parameters
    ----------
    x, y : array-like
        Input data.
    n_permutations : int, optional
        Number of permutations for p-value calculation.
    random_state : int or None
        Random seed.
    **ransac_kwargs :
        Extra keyword arguments for sklearn.linear_model.RANSACRegressor.

    Returns
    -------
    slope : float
        Estimated slope from the inlier model.
    intercept : float
        Estimated intercept.
    p_value : float
        Two-sided permutation p-value for H0: slope = 0.
    """
    rng = np.random.default_rng(random_state)
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float)

    # Fit RANSAC with linear base estimator
    base_model = LinearRegression()
    model = RANSACRegressor(
                            random_state=random_state,
                            **ransac_kwargs)
    model.fit(x, y)
    slope = model.estimator_.coef_[0]
    intercept = model.estimator_.intercept_

    # Null distribution via permutation
    perm_slopes = np.empty(n_permutations)
    for i in range(n_permutations):
        y_perm = rng.permutation(y)
        perm_model = RANSACRegressor(
                                     random_state=random_state,
                                     **ransac_kwargs)
        perm_model.fit(x, y_perm)
        perm_slopes[i] = perm_model.estimator_.coef_[0]

    # Two-sided p-value
    p_value = np.mean(np.abs(perm_slopes) >= np.abs(slope))

    return slope, intercept, p_value




def huber_with_pvalue(x, y, n_permutations=1000, random_state=None, **huber_kwargs):
    """
    Huber regression with permutation-based p-value for slope != 0.

    Parameters
    ----------
    x, y : array-like
        Input data.
    n_permutations : int, optional
        Number of permutations for p-value calculation.
    random_state : int or None
        Random seed.
    **huber_kwargs : 
        Extra keyword arguments for sklearn.linear_model.HuberRegressor.

    Returns
    -------
    slope : float
        Estimated slope.
    intercept : float
        Estimated intercept.
    p_value : float
        Two-sided p-value for H0: slope = 0.
    """
    rng = np.random.default_rng(random_state)
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float)

    # Fit Huber regression
    model = HuberRegressor(**huber_kwargs).fit(x, y)
    slope = model.coef_[0]
    intercept = model.intercept_

    # Null distribution via permutation
    perm_slopes = np.empty(n_permutations)
    for i in range(n_permutations):
        y_perm = rng.permutation(y)
        perm_model = HuberRegressor(**huber_kwargs).fit(x, y_perm)
        perm_slopes[i] = perm_model.coef_[0]

    # Two-sided p-value
    p_value = np.mean(np.abs(perm_slopes) >= np.abs(slope))

    return slope, intercept, p_value



def theil_sen_with_pvalue(x, y, n_permutations=10000, random_state=None):
    """
    Theil–Sen regression with permutation-based p-value for slope != 0.
    
    Parameters
    ----------
    x, y : array-like
        Input data.
    n_permutations : int, optional
        Number of permutations for p-value calculation.
    random_state : int or None
        Random seed.
        
    Returns
    -------
    slope : float
        Theil–Sen slope estimate.
    intercept : float
        Intercept estimate.
    p_value : float
        Two-sided p-value for H0: slope = 0.
    """
    rng = np.random.default_rng(random_state)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Theil–Sen fit
    slope, intercept, _, _ = theilslopes(y, x)

    # Null distribution under H0: slope = 0
    perm_slopes = np.empty(n_permutations)
    for i in range(n_permutations):
        y_perm = rng.permutation(y)
        perm_slopes[i], _, _, _ = theilslopes(y_perm, x)

    # Two-sided p-value
    p_value = np.mean(np.abs(perm_slopes) >= np.abs(slope))

    return slope, intercept, p_value




# assuming CombDB is your DataFrame, CyKeys is your list of "x" columns,
# and Dict1 maps each x to its corresponding "y" column name
results = []

for k in tqdm(CyKeys):
    x = CombDB[k]
    y = CombDB[Dict1[k]]
    # drop any rows where either is NaN
    mask = x.notna() & y.notna()
    x_clean = x[mask]
    y_clean = y[mask]
    
    # compute r and p
    r, p = spearmanr(x_clean, y_clean)
#    slope, intercept, p_value = theil_sen_with_pvalue(x_clean.values.reshape(-1,1), y_clean.values)
 
    
    results.append({
        'x': k,
        'y': Dict1[k],
        'r': r,
        'p_value': p,
        # 'TH':slope,
        # 'p TH':p_value,

    })

# turn into a DataFrame for easy viewing
results_df = pd.DataFrame(results)
print(results_df)
results_df.to_csv("Correlations_EPINUC_CyTOF.csv",index=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, linregress
from tqdm import tqdm

results = []

for k in tqdm(CyKeys):
    x = CombDB[k]
    y = CombDB[Dict1[k]]

    # drop NaNs
    mask = x.notna() & y.notna()
    x_clean = x[mask]
    y_clean = y[mask]

    # need at least 2 points to fit a line
    if len(x_clean) < 2:
        continue

    # Spearman correlation
    r_spear, p_spear = spearmanr(x_clean, y_clean)

    # OLS regression (y = slope*x + intercept)
    lr = linregress(x_clean.values, y_clean.values)
    slope, intercept, p_slope = lr.slope, lr.intercept, lr.pvalue

    results.append({
        'x': k,
        'y': Dict1[k],
        'Spearman_r': r_spear,
        'Spearman_p': p_spear,
        'OLS_slope': slope,
        'OLS_intercept': intercept,
        'OLS_p_slope': p_slope,
        'OLS_r_pearson': lr.rvalue,
        'OLS_stderr_slope': lr.stderr,
        'OLS_stderr_intercept': lr.intercept_stderr if hasattr(lr, 'intercept_stderr') else np.nan
    })

    # Plot points and OLS regression line
    plt.figure(figsize=(5, 4))
    plt.scatter(x_clean, y_clean, alpha=0.7, label="Data",color='k')

    x_range = np.linspace(x_clean.min(), x_clean.max(), 100)
    y_pred = slope * x_range + intercept
    plt.plot(x_range, y_pred, lw=2,
             label=f"OLS fit\nslope={slope:.3f}, p={p_slope:.3g}",color='k')

    plt.xlabel(k)
    plt.ylabel(Dict1[k])
    plt.title(f"{k} vs {Dict1[k]}")
    # plt.legend()
    plt.tight_layout()
    plt.show()

# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df)


In [ ]:
import numpy as np
from scipy.stats import theilslopes

def theil_sen_with_pvalue(
    x, y,
    n_permutations=10000,
    alpha=0.95,
    n_boot=1000,
    x_grid=None,            # optional array for returning a band
    random_state=None
):
    """
    Theil–Sen regression with:
      - permutation-based p-value for H0: slope = 0
      - bootstrap CIs for BOTH slope and intercept
      - optional joint (slope, intercept) CI band over x_grid

    Parameters
    ----------
    x, y : array-like
    n_permutations : int
        For permutation p-value of slope != 0 (permute y against x).
    alpha : float
        Confidence level for CIs (default 0.95).
    n_boot : int
        Bootstrap resamples for joint (slope, intercept) uncertainty.
    x_grid : array-like or None
        If provided, returns (y_lower, y_upper) for this grid
        using the bootstrap joint distribution.
    random_state : int or None

    Returns
    -------
    slope : float
    intercept : float
    p_value : float
    slope_lo : float
    slope_hi : float
    intercept_lo : float
    intercept_hi : float
    band_lower : np.ndarray or None
    band_upper : np.ndarray or None
    """
    rng = np.random.default_rng(random_state)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Base Theil–Sen fit (SciPy also gives a slope CI, we still compute joint CI via bootstrap)
    slope, intercept, slope_lo_sci, slope_hi_sci = theilslopes(y, x, alpha=alpha)

    # Permutation p-value for slope != 0
    perm_slopes = np.empty(n_permutations, dtype=float)
    for i in range(n_permutations):
        y_perm = rng.permutation(y)
        ps, _, _, _ = theilslopes(y_perm, x, alpha=alpha)
        perm_slopes[i] = ps
    p_value = np.mean(np.abs(perm_slopes) >= np.abs(slope))

    # Bootstrap for joint (slope, intercept) distribution
    boot_slopes = []
    boot_intercepts = []
    n = len(x)
    if n_boot > 0 and n >= 3:
        for _ in range(n_boot):
            idx = rng.integers(0, n, n)
            xb, yb = x[idx], y[idx]
            try:
                s, i, _, _ = theilslopes(yb, xb, alpha=alpha)
                if np.isfinite(s) and np.isfinite(i):
                    boot_slopes.append(s)
                    boot_intercepts.append(i)
            except Exception:
                # skip degenerate resamples
                continue

    boot_slopes = np.asarray(boot_slopes)
    boot_intercepts = np.asarray(boot_intercepts)

    if boot_slopes.size >= 10:  # need enough successful resamples
        q_lo = (1 - alpha) / 2 * 100
        q_hi = (1 + alpha) / 2 * 100
        slope_lo = np.percentile(boot_slopes, q_lo)
        slope_hi = np.percentile(boot_slopes, q_hi)
        intercept_lo = np.percentile(boot_intercepts, q_lo)
        intercept_hi = np.percentile(boot_intercepts, q_hi)
    else:
        # Fallback: use SciPy slope CI and no intercept CI
        slope_lo, slope_hi = slope_lo_sci, slope_hi_sci
        intercept_lo = np.nan
        intercept_hi = np.nan

    band_lower = band_upper = None
    if x_grid is not None and boot_slopes.size > 0:
        # Build band from joint draws: y = s*x + i for each bootstrap draw
        xg = np.asarray(x_grid, float)
        y_boot = np.outer(boot_slopes, xg) + boot_intercepts[:, None]
        q_lo = (1 - alpha) / 2 * 100
        q_hi = (1 + alpha) / 2 * 100
        band_lower = np.nanpercentile(y_boot, q_lo, axis=0)
        band_upper = np.nanpercentile(y_boot, q_hi, axis=0)

    return (slope, intercept, p_value,
            slope_lo, slope_hi, intercept_lo, intercept_hi,
            band_lower, band_upper)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from tqdm import tqdm

# Collect results here
results = []

alpha = 0.95        # 95% CI
n_perm = 10000
n_boot = 1000

for k in tqdm(CyKeys):
    x = CombDB[k]
    y = CombDB[Dict1[k]]

    # drop NaNs
    mask = x.notna() & y.notna()
    x_clean = x[mask].values
    y_clean = y[mask].values

    if x_clean.size < 3:
        continue  # need enough points

    # Spearman correlation
    r, p = spearmanr(x_clean, y_clean)

    # Range for plotting + pass to TS helper so it returns the band
    x_grid = np.linspace(x_clean.min(), x_clean.max(), 200)

    (slope, intercept, p_slope,
     slope_lo, slope_hi, intercept_lo, intercept_hi,
     y_lo, y_hi) = theil_sen_with_pvalue(
        x_clean, y_clean,
        n_permutations=n_perm,
        alpha=alpha,
        n_boot=n_boot,
        x_grid=x_grid,
        random_state=0
    )

    results.append({
        'x': k,
        'y': Dict1[k],
        'Spearman_r': r,
        'Spearman_p': p,
        'TH_slope': slope,
        'TH_intercept': intercept,
        'TH_p_slope': p_slope,
        f'TH_slope_CI_{int(alpha*100)}_lo': slope_lo,
        f'TH_slope_CI_{int(alpha*100)}_hi': slope_hi,
        f'TH_intercept_CI_{int(alpha*100)}_lo': intercept_lo,
        f'TH_intercept_CI_{int(alpha*100)}_hi': intercept_hi
    })

    # Predicted mean line
    y_pred = slope * x_grid + intercept

    # Plot
    plt.figure(figsize=(5, 4))
    plt.scatter(x_clean, y_clean, alpha=0.7, color='k', label="Data")

    plt.plot(x_grid, y_pred, color='black', lw=2,
             label=(f"Theil–Sen fit\n"
                    f"slope={slope:.3f} "
                    f"(CI {int(alpha*100)}%: {slope_lo:.3f},{slope_hi:.3f}), "
                    f"p={p_slope:.3g}"))

    if y_lo is not None and y_hi is not None:
        plt.fill_between(x_grid, y_lo, y_hi, color='gray', alpha=0.25,
                         label=f"{int(alpha*100)}% joint CI band")

    plt.xlabel(k)
    plt.ylabel(Dict1[k])
    plt.title(f"{k} vs {Dict1[k]}")
    plt.legend(bbox_to_anchor=(1,1))
    plt.tight_layout()
    plt.savefig(f"Plots/{k}.pdf",dpi=200,bbox_inches='tight')
    plt.show()

# Results table
results_df = pd.DataFrame(results)
print(results_df)
results_df.to_csv("Plots/Correlations_EPINUC_CyTOF.csv",index=False)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from tqdm import tqdm

# Assuming theil_sen_with_pvalue is already defined from earlier
results = []

for k in tqdm(CyKeys):
    x = CombDB[k]
    y = CombDB[Dict1[k]]

    # drop NaNs
    mask = x.notna() & y.notna()
    x_clean = x[mask]
    y_clean = y[mask]

    # Spearman correlation
    r, p = spearmanr(x_clean, y_clean)

    # Theil–Sen slope, intercept, p-value
    slope, intercept, p_value = theil_sen_with_pvalue(
        x_clean.values,  # 1D array is fine here
        y_clean.values
    )

    results.append({
        'x': k,
        'y': Dict1[k],
        'r': r,
        'p_value': p,
        'TH': slope,
        'p TH': p_value
    })

    # Plot points and regression line
    plt.figure(figsize=(5, 4))
    plt.scatter(x_clean, y_clean, alpha=0.7, label="Data points",color='k')
    
    # Regression line
    x_range = np.linspace(x_clean.min(), x_clean.max(), 100)
    y_pred = slope * x_range + intercept
    plt.plot(x_range, y_pred, color='black', lw=2,
             label=f"Theil–Sen fit\nslope={slope:.3f}, p={p_value:.3g}")
    
    plt.xlabel(k)
    plt.ylabel(Dict1[k])
    plt.title(f"{k} vs {Dict1[k]}")
    #plt.legend()
    plt.tight_layout()
    plt.show()

# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for k in CyKeys:
    fig, ax = plt.subplots()
    # draw the scatter
    sns.scatterplot(data=CombDB, x=k, y=Dict1[k], ax=ax)
    
    # grab the x- and y-values, and the labels you want
    x = CombDB[k]
    y = CombDB[Dict1[k]]
    labels = CombDB.index  # must be same length/order as CombDB
    
    # annotate each point
    for xi, yi, lab in zip(x, y, labels):
        ax.annotate(
            str(lab),
            (xi, yi),
            textcoords="offset points",
            xytext=(5, 5),      # shift text 5pts right, 5pts up
            ha="left",          # horizontal alignment
            fontsize=8
        )
    
    ax.set_title(f"{k} vs {Dict1[k]}")
    plt.savefig(f"Plots/Scatter_{k}.png",dpi=200,bbox_inches='tight')
    plt.show()


In [ ]:
# pip install numpy pandas scipy matplotlib
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# -------- Deming regression (errors in both variables) ----------
def deming_regression(x, y, delta=1.0, ci=False, n_boot=2000, random_state=0):
    """
    Deming regression with optional bootstrap CIs.
    delta = sigma_y^2 / sigma_x^2 (set 1.0 if unknown).
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]

    xbar, ybar = x.mean(), y.mean()
    s_xx = np.var(x, ddof=1)
    s_yy = np.var(y, ddof=1)
    s_xy = np.cov(x, y, ddof=1)[0, 1]

    beta = ((s_yy - delta*s_xx) + np.sqrt((s_yy - delta*s_xx)**2 + 4*delta*s_xy**2)) / (2*s_xy)
    alpha = ybar - beta*xbar

    out = {"alpha": alpha, "beta": beta}

    if not ci:
        return out

    rng = np.random.default_rng(random_state)
    boots = []
    n = len(x)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        xb, yb = x[idx], y[idx]
        xbar_b, ybar_b = xb.mean(), yb.mean()
        s_xx_b = np.var(xb, ddof=1)
        s_yy_b = np.var(yb, ddof=1)
        s_xy_b = np.cov(xb, yb, ddof=1)[0, 1]
        beta_b = ((s_yy_b - delta*s_xx_b) + np.sqrt((s_yy_b - delta*s_xx_b)**2 + 4*delta*s_xy_b**2)) / (2*s_xy_b)
        alpha_b = ybar_b - beta_b*xbar_b
        boots.append((alpha_b, beta_b))
    boots = np.array(boots)
    out["alpha_ci"] = tuple(np.quantile(boots[:,0], [0.025, 0.975]))
    out["beta_ci"]  = tuple(np.quantile(boots[:,1], [0.025, 0.975]))
    return out

# -------- Bland–Altman (LoA) ----------
def bland_altman(x, y):
    """
    NaN-safe Bland–Altman stats (bias and 95% LoA).
    Returns dict and (mean, diff) arrays for plotting.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    mean = (x + y) / 2
    diff = x - y
    bias = diff.mean()
    sd = diff.std(ddof=1)
    loa_lower = bias - 1.96*sd
    loa_upper = bias + 1.96*sd
    return {"bias": bias, "loa_lower": loa_lower, "loa_upper": loa_upper, "n_used": len(x)}, mean, diff

# -------- One-call wrapper ----------
def analyze_agreement(x, y, label_x="Method 1", label_y="Method 2",
                      delta=1.0, title_suffix="", show_plots=True, deming_ci=False):
    """
    Runs Bland–Altman, Spearman, Deming. NaN-safe.
    Returns a dict summary.
    """
    # Spearman (monotonic association)
    rho, pval = stats.spearmanr(x, y, nan_policy="omit")

    # Deming (errors-in-variables)
    dem = deming_regression(x, y, delta=delta, ci=deming_ci)

    # Bland–Altman
    ba, mean_xy, diff_xy = bland_altman(x, y)

    summary = {
        "spearman_rho": rho, "spearman_p": pval,
        "deming_alpha": dem["alpha"], "deming_beta": dem["beta"],
        "deming_alpha_ci": dem.get("alpha_ci"), "deming_beta_ci": dem.get("beta_ci"),
        "bias": ba["bias"], "loa_lower": ba["loa_lower"], "loa_upper": ba["loa_upper"],
        "n_used": ba["n_used"]
    }

    if show_plots:
        # --- Bland–Altman plot ---
        plt.figure()
        plt.scatter(mean_xy, diff_xy, alpha=0.7)
        plt.axhline(ba["bias"], linestyle="--", label=f"Bias={ba['bias']:.3g}")
        plt.axhline(ba["loa_upper"], linestyle="--", label=f"+1.96 SD={ba['loa_upper']:.3g}")
        plt.axhline(ba["loa_lower"], linestyle="--", label=f"-1.96 SD={ba['loa_lower']:.3g}")
        plt.xlabel("Mean of methods")
        plt.ylabel(f"Difference ({label_x} - {label_y})")
        plt.title(f"Bland–Altman {title_suffix}".strip())
        plt.legend()
        plt.show()

        # --- Deming regression plot ---
        x_arr = np.asarray(x, float); y_arr = np.asarray(y, float)
        mask = np.isfinite(x_arr) & np.isfinite(y_arr)
        x_arr, y_arr = x_arr[mask], y_arr[mask]
        xv = np.linspace(x_arr.min(), x_arr.max(), 200)
        yv = dem["alpha"] + dem["beta"]*xv

        plt.figure()
        plt.scatter(x_arr, y_arr, alpha=0.7, label="Data")
        plt.plot(xv, yv, label=f"Deming: y={dem['alpha']:.2f}+{dem['beta']:.2f}x")
        plt.xlabel(label_x); plt.ylabel(label_y)
        plt.title(f"Deming Regression {title_suffix}".strip())
        plt.legend()
        plt.show()

    return summary


In [ ]:
# pip install numpy pandas scipy matplotlib
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# -------- Passing–Bablok regression (nonparametric, robust) ----------
def passing_bablok(x, y, ci=False, n_boot=2000, random_state=0):
    """
    Passing–Bablok regression with optional bootstrap CIs.
    Robust, nonparametric method-comparison regression.

    Returns:
        dict with keys:
            alpha: intercept
            beta: slope
            alpha_ci: (lo, hi) if ci=True
            beta_ci: (lo, hi) if ci=True
    Notes:
        - Pairs with x_j == x_i are excluded (vertical lines).
        - Intercept is median(y - beta*x).
        - Complexity ~ O(n^2) for slope computation.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 2:
        raise ValueError("Need at least two paired points for Passing–Bablok.")

    # --- Compute all pairwise slopes for i < j, excluding verticals ---
    slopes = []
    for i in range(n - 1):
        dx = x[i+1:] - x[i]
        dy = y[i+1:] - y[i]
        valid = dx != 0
        if np.any(valid):
            slopes.extend((dy[valid] / dx[valid]).tolist())

    if len(slopes) == 0:
        # Degenerate case: all x equal -> undefined slope
        raise ValueError("Passing–Bablok undefined: all x values are identical (vertical).")

    slopes = np.array(slopes)
    beta = np.median(slopes)
    alpha = np.median(y - beta * x)

    out = {"alpha": float(alpha), "beta": float(beta)}

    if not ci:
        return out

    # --- Bootstrap percentile CI (resample pairs) ---
    rng = np.random.default_rng(random_state)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)  # resample paired observations with replacement
        xb, yb = x[idx], y[idx]

        # recompute PB on the bootstrap sample
        sb = []
        nb = len(xb)
        for i in range(nb - 1):
            dx = xb[i+1:] - xb[i]
            dy = yb[i+1:] - yb[i]
            valid = dx != 0
            if np.any(valid):
                sb.extend((dy[valid] / dx[valid]).tolist())

        if len(sb) == 0:
            # fallback: skip iteration if degenerate
            continue

        sb = np.array(sb)
        beta_b = np.median(sb)
        alpha_b = np.median(yb - beta_b * xb)
        boots.append((alpha_b, beta_b))

    boots = np.array(boots)
    if boots.size == 0:
        # If all boots degenerated, return NaN CIs
        out["alpha_ci"] = (np.nan, np.nan)
        out["beta_ci"]  = (np.nan, np.nan)
    else:
        out["alpha_ci"] = tuple(np.quantile(boots[:, 0], [0.025, 0.975]))
        out["beta_ci"]  = tuple(np.quantile(boots[:, 1], [0.025, 0.975]))

    return out

# -------- Bland–Altman (LoA) ----------
def bland_altman(x, y):
    """
    NaN-safe Bland–Altman stats (bias and 95% LoA).
    Returns dict and (mean, diff) arrays for plotting.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    mean = (x + y) / 2
    diff = x - y
    bias = diff.mean()
    sd = diff.std(ddof=1)
    loa_lower = bias - 1.96*sd
    loa_upper = bias + 1.96*sd
    return {"bias": bias, "loa_lower": loa_lower, "loa_upper": loa_upper, "n_used": len(x)}, mean, diff

# -------- One-call wrapper ----------
def analyze_agreement(x, y, label_x="Method 1", label_y="Method 2",
                      title_suffix="", show_plots=True, pb_ci=False):
    """
    Runs Bland–Altman, Spearman, Passing–Bablok. NaN-safe.
    Returns a dict summary.

    Replaces Deming with Passing–Bablok (robust, nonparametric).
    """
    # Spearman (monotonic association)
    rho, pval = stats.spearmanr(x, y, nan_policy="omit")

    # Passing–Bablok (robust method-comparison regression)
    pb = passing_bablok(x, y, ci=pb_ci)

    # Bland–Altman
    ba, mean_xy, diff_xy = bland_altman(x, y)

    summary = {
        "spearman_rho": rho, "spearman_p": pval,
        "pb_alpha": pb["alpha"], "pb_beta": pb["beta"],
        "pb_alpha_ci": pb.get("alpha_ci"), "pb_beta_ci": pb.get("beta_ci"),
        "bias": ba["bias"], "loa_lower": ba["loa_lower"], "loa_upper": ba["loa_upper"],
        "n_used": ba["n_used"]
    }

    if show_plots:
        # --- Bland–Altman plot ---
        plt.figure()
        plt.scatter(mean_xy, diff_xy, alpha=0.7)
        plt.axhline(ba["bias"], linestyle="--", label=f"Bias={ba['bias']:.3g}")
        plt.axhline(ba["loa_upper"], linestyle="--", label=f"+1.96 SD={ba['loa_upper']:.3g}")
        plt.axhline(ba["loa_lower"], linestyle="--", label=f"-1.96 SD={ba['loa_lower']:.3g}")
        plt.xlabel("Mean of methods")
        plt.ylabel(f"Difference ({label_x} - {label_y})")
        plt.title(f"Bland–Altman {title_suffix}".strip())
        plt.legend()
        plt.show()

        # --- Passing–Bablok regression plot ---
        x_arr = np.asarray(x, float); y_arr = np.asarray(y, float)
        mask = np.isfinite(x_arr) & np.isfinite(y_arr)
        x_arr, y_arr = x_arr[mask], y_arr[mask]
        xv = np.linspace(x_arr.min(), x_arr.max(), 200)
        yv = pb["alpha"] + pb["beta"]*xv

        plt.figure()
        plt.scatter(x_arr, y_arr, alpha=0.7, label="Data")
        plt.plot(xv, yv, label=f"Passing–Bablok: y={pb['alpha']:.2f}+{pb['beta']:.2f}x")
        plt.xlabel(label_x); plt.ylabel(label_y)
        plt.title(f"Passing–Bablok Regression {title_suffix}".strip())
        plt.legend()
        plt.show()

    return summary


In [ ]:
# pip install numpy pandas scipy matplotlib pingouin
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# --- Pingouin for shift plot ---
try:
    import pingouin as pg
except ImportError as e:
    raise ImportError("Please install pingouin first: pip install pingouin") from e


# -------- Passing–Bablok regression (nonparametric, robust) ----------
def passing_bablok(x, y, ci=False, n_boot=2000, random_state=0):
    """
    Passing–Bablok regression with optional bootstrap CIs.
    Robust, nonparametric method-comparison regression.

    Returns:
        dict with keys:
            alpha: intercept
            beta: slope
            alpha_ci: (lo, hi) if ci=True
            beta_ci: (lo, hi) if ci=True
    Notes:
        - Pairs with x_j == x_i are excluded (vertical lines).
        - Intercept is median(y - beta*x).
        - Complexity ~ O(n^2) for slope computation.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 2:
        raise ValueError("Need at least two paired points for Passing–Bablok.")

    # --- Compute all pairwise slopes for i < j, excluding verticals ---
    slopes = []
    for i in range(n - 1):
        dx = x[i+1:] - x[i]
        dy = y[i+1:] - y[i]
        valid = dx != 0
        if np.any(valid):
            slopes.extend((dy[valid] / dx[valid]).tolist())

    if len(slopes) == 0:
        # Degenerate case: all x equal -> undefined slope
        raise ValueError("Passing–Bablok undefined: all x values are identical (vertical).")

    slopes = np.array(slopes)
    beta = np.median(slopes)
    alpha = np.median(y - beta * x)

    out = {"alpha": float(alpha), "beta": float(beta)}

    if not ci:
        return out

    # --- Bootstrap percentile CI (resample pairs) ---
    rng = np.random.default_rng(random_state)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)  # resample paired observations with replacement
        xb, yb = x[idx], y[idx]

        # recompute PB on the bootstrap sample
        sb = []
        nb = len(xb)
        for i in range(nb - 1):
            dx = xb[i+1:] - xb[i]
            dy = yb[i+1:] - yb[i]
            valid = dx != 0
            if np.any(valid):
                sb.extend((dy[valid] / dx[valid]).tolist())

        if len(sb) == 0:
            # skip degenerate bootstrap sample
            continue

        sb = np.array(sb)
        beta_b = np.median(sb)
        alpha_b = np.median(yb - beta_b * xb)
        boots.append((alpha_b, beta_b))

    boots = np.array(boots)
    if boots.size == 0:
        # If all boots degenerated, return NaN CIs
        out["alpha_ci"] = (np.nan, np.nan)
        out["beta_ci"]  = (np.nan, np.nan)
    else:
        out["alpha_ci"] = tuple(np.quantile(boots[:, 0], [0.025, 0.975]))
        out["beta_ci"]  = tuple(np.quantile(boots[:, 1], [0.025, 0.975]))

    return out


# -------- Bland–Altman (LoA) ----------
def bland_altman(x, y):
    """
    NaN-safe Bland–Altman stats (bias and 95% LoA).
    Returns dict and (mean, diff) arrays for plotting.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    mean = (x + y) / 2
    diff = x - y
    bias = diff.mean()
    sd = diff.std(ddof=1)
    loa_lower = bias - 1.96 * sd
    loa_upper = bias + 1.96 * sd
    return {"bias": bias, "loa_lower": loa_lower, "loa_upper": loa_upper, "n_used": len(x)}, mean, diff


# -------- One-call wrapper (Passing–Bablok + Bland–Altman + Shift plot) ----------
def analyze_agreement(
    x, y, label_x="Method 1", label_y="Method 2",
    title_suffix="", show_plots=True, pb_ci=False,
    # Shift plot options
    shift_plot=True, shift_n_boot=2000, shift_confidence=0.95, shift_seed=42,
    shift_percentiles=None, shift_show_median=True, shift_violin=True
):
    """
    Runs Bland–Altman, Spearman, Passing–Bablok, and (optionally) a Pingouin shift plot.
    NaN-safe. Returns a dict summary.
    """
    # Spearman (monotonic association)
    rho, pval = stats.spearmanr(x, y, nan_policy="omit")

    # Passing–Bablok (robust method-comparison regression)
    pb = passing_bablok(x, y, ci=pb_ci)

    # Bland–Altman
    ba, mean_xy, diff_xy = bland_altman(x, y)

    summary = {
        "spearman_rho": rho, "spearman_p": pval,
        "pb_alpha": pb["alpha"], "pb_beta": pb["beta"],
        "pb_alpha_ci": pb.get("alpha_ci"), "pb_beta_ci": pb.get("beta_ci"),
        "bias": ba["bias"], "loa_lower": ba["loa_lower"], "loa_upper": ba["loa_upper"],
        "n_used": ba["n_used"]
    }

    if show_plots:
        # Prepare clean arrays once
        x_arr = np.asarray(x, float); y_arr = np.asarray(y, float)
        mask = np.isfinite(x_arr) & np.isfinite(y_arr)
        x_arr, y_arr = x_arr[mask], y_arr[mask]

        # --- Bland–Altman plot ---
        plt.figure()
        plt.scatter(mean_xy, diff_xy, alpha=0.7)
        plt.axhline(ba["bias"], linestyle="--", label=f"Bias={ba['bias']:.3g}")
        plt.axhline(ba["loa_upper"], linestyle="--", label=f"+1.96 SD={ba['loa_upper']:.3g}")
        plt.axhline(ba["loa_lower"], linestyle="--", label=f"-1.96 SD={ba['loa_lower']:.3g}")
        plt.xlabel("Mean of methods")
        plt.ylabel(f"Difference ({label_x} - {label_y})")
        plt.title(f"Bland–Altman {title_suffix}".strip())
        plt.legend()
        plt.show()

        # --- Passing–Bablok regression plot ---
        xv = np.linspace(x_arr.min(), x_arr.max(), 200)
        yv = pb["alpha"] + pb["beta"] * xv
        plt.figure()
        plt.scatter(x_arr, y_arr, alpha=0.7, label="Data")
        plt.plot(xv, yv, label=f"Passing–Bablok: y={pb['alpha']:.2f}+{pb['beta']:.2f}x")
        plt.xlabel(label_x); plt.ylabel(label_y)
        plt.title(f"Passing–Bablok Regression {title_suffix}".strip())
        plt.legend()
        plt.show()

        # --- Shift plot (Pingouin) ---
        if shift_plot:
            kwargs = {}
            if shift_percentiles is not None:
                kwargs["percentiles"] = shift_percentiles

            fig = pg.plot_shift(
                x_arr, y_arr,
                paired=True,
                n_boot=shift_n_boot,
                confidence=shift_confidence,
                seed=shift_seed,
                show_median=shift_show_median,
                violin=shift_violin,
                **kwargs
            )
            # Add titles / labels
            fig.suptitle(f"Shift Plot {title_suffix}".strip())
            # fig.axes[0] = distributions; fig.axes[1] = shift function (usually)
            if fig.axes:
                fig.axes[0].set_xlabel(label_x)
                fig.axes[0].set_ylabel(label_y)
            plt.tight_layout()
            plt.show()

    return summary





In [ ]:
import numpy as np
from scipy.stats import pearsonr, spearmanr

def _corr_core(x, y, method):
    if method == "pearson":
        r = pearsonr(x, y)[0]
    elif method == "spearman":
        r = spearmanr(x, y)[0]
    else:
        raise ValueError("method must be 'pearson' or 'spearman'")
    if np.isfinite(r):
        r = float(np.clip(r, -1.0, 1.0))
    return r

def _is_valid_pair(a, b, min_unique=2):
    return (np.unique(a).size >= min_unique) and (np.unique(b).size >= min_unique)

def correlation_with_ci_safe(
    x, y,
    n_boot=5000,
    n_perm=5000,
    method="pearson",
    random_state=None,
    min_unique=2,
    max_boot_attempt_factor=50,
    use_fisher_fallback=True
):
    """
    Correlation with bootstrap CI and permutation test.
    - Removes NaNs pairwise
    - Skips invalid resamples (constant arrays)
    - Uses +1 correction for permutation p-values

    Returns dict with:
      observed_corr, ci_95 (tuple), perm_p_value,
      boot_valid, perm_valid, notes
    """
    rng = np.random.default_rng(random_state)
    x, y = np.asarray(x), np.asarray(y)

    # 1. Remove NaNs pairwise
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 3:
        raise ValueError("Need at least 3 valid pairs after removing NaNs.")

    # 2. Observed correlation
    obs = _corr_core(x, y, method)

    # -------------------
    # 3. Bootstrap CI
    # -------------------
    boot_vals = []
    target = n_boot
    attempts, max_attempts = 0, max_boot_attempt_factor * max(1, target)

    while len(boot_vals) < target and attempts < max_attempts:
        attempts += 1
        idx = rng.integers(0, n, n)
        xb, yb = x[idx], y[idx]
        if not _is_valid_pair(xb, yb, min_unique=min_unique):
            continue
        r = _corr_core(xb, yb, method)
        if np.isfinite(r):
            boot_vals.append(r)

    boot_valid = len(boot_vals)
    notes = []
    if boot_valid < max(100, 0.25 * target):
        notes.append(f"Only {boot_valid} valid bootstrap resamples collected.")
        if use_fisher_fallback and method == "pearson" and np.isfinite(obs):
            # Fisher-z fallback
            z = np.arctanh(np.clip(obs, -0.999999, 0.999999))
            se = 1 / np.sqrt(n - 3)
            ci = (np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se))
            notes.append("Used Fisher-z analytic CI (Pearson fallback).")
        else:
            ci = (np.nan, np.nan) if boot_valid == 0 else tuple(np.percentile(boot_vals, [2.5, 97.5]))
    else:
        ci = tuple(np.percentile(boot_vals, [2.5, 97.5]))

    # -------------------
    # 4. Permutation test
    # -------------------
    perm_extreme = 0
    perm_valid = 0
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        if not _is_valid_pair(x, y_perm, min_unique=min_unique):
            continue
        r = _corr_core(x, y_perm, method)
        if not np.isfinite(r):
            continue
        perm_valid += 1
        if abs(r) >= abs(obs):
            perm_extreme += 1

    if perm_valid == 0:
        pval = np.nan
        notes.append("No valid permutations (y became constant too often).")
    else:
        # +1 correction avoids zero p-values
        pval = (perm_extreme + 1) / (perm_valid + 1)

    return {
        "observed_corr": obs,
        "ci_95": ci,
        "perm_p_value": pval,
        "boot_valid": boot_valid,
        "perm_valid": perm_valid,
        "notes": "; ".join(notes) if notes else ""
    }



In [ ]:
from tqdm import tqdm

In [ ]:
L=[]
for k in tqdm(CyKeys):
    x=CombDB[k].values
    y=CombDB[Dict1[k]].values
    out=correlation_with_ci_safe(x, y, n_boot=5000, n_perm=5000, method="spearman", random_state=None)
    L.append(out)    

In [ ]:
pd.DataFrame(L,index=CyKeys)

In [ ]:
x

In [ ]:
y